<a href="https://colab.research.google.com/github/hilalhmohammedsalim-sys/Study_Buddy_Chatbot_Project/blob/main/Study_buddy_Chatbot_corrected_Project1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📚 PDF Study Buddy — RAG + Agentic AI + Gen AI

**Corrected and re-ordered notebook.** Run it **top to bottom** (Runtime → Run all works too; it will stop twice to ask for your PDF and your Pinecone key).
After a Colab restart, run everything again from the top. The Ollama server, the models and all variables are wiped on restart.

**Tip:** switch to a GPU first (Runtime → Change runtime type → T4 GPU). Answers become much faster.

## How the whole thing works

```
PDF → clean text → cut into chunks → turn chunks into numbers (embeddings) → store in Pinecone

Question → Pinecone finds 12 close chunks → reranker keeps the best 4
        → LLM writes the answer using only those chunks (or says "I don't know")

Agent: your message → router picks a tool → answer | quiz | summary | show-answers
```

| Part | What it does |
|---|---|
| 1. Setup | Installs tools, starts the local LLM (Ollama), loads the embedding model, connects to Pinecone |
| 2. Load PDF | Reads and cleans the text, removes the references section |
| 3. Chunks | Splits the text into small pieces that keep their page number |
| 4. Vector DB | Stores the chunks in Pinecone |
| 5. Search | Finds and reranks the chunks that match a question |
| 6. Answer | The RAG answer function with an "I don't know" guard |
| 7. Evaluation | 36 test questions that measure how accurate the bot really is |
| 8. Quiz tool | Makes multiple-choice questions and checks them |
| 9. Summary tool | Summarizes the whole PDF or one topic |
| 10. Agent | The router that picks the right tool |
| 11. Web page | A Gradio chat page |

# Part 1 — Setup

### 1.1 Install the Python libraries
`pypdf` reads PDFs. `sentence-transformers` makes embeddings and the reranker. `pinecone` is the vector database client. `ollama` talks to the local LLM.
**Check:** it finishes without red errors.

In [ ]:
!pip install -q pypdf sentence-transformers pinecone ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.1/405.1 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 21.5 MB/s eta 0:00:00


### 1.2 Check the libraries imported
**Check:** you see three version numbers and the ✅. If you see `ModuleNotFoundError`, re-run 1.1.

In [ ]:
import pypdf, sentence_transformers, pinecone, ollama
print("pypdf:", pypdf.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("pinecone:", pinecone.__version__)
print("All libraries imported OK ✅")

pypdf: 6.21.1
sentence-transformers: 5.7.0
pinecone: 10.0.0
All libraries imported OK ✅


### 1.3 Install and start Ollama (the local LLM server)
**Correction:** your last run printed "Unable to detect NVIDIA/AMD GPU", so Ollama probably ran on the CPU. The installer needs the tools `pciutils` and `lshw` to find the GPU, so they are now installed first. To confirm it worked, the cell in 1.4 now prints the GPU name and which processor Ollama uses (the `PROCESSOR` column of `ollama ps` should say GPU).
**Correction:** the old version waited a fixed 8 seconds and hoped the server was ready. This version keeps asking the server until it really answers (up to 30 seconds).
**Check:** `Ollama server is up ✅`. If it prints the ⚠️ message instead, run the cell again.

In [ ]:
!apt-get install -y -q zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, urllib.request

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434", timeout=1)
        print("Ollama server is up ✅")
        break
    except Exception:
        time.sleep(1)
else:
    print("⚠️ Ollama did not answer in 30 seconds. Run this cell again.")

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Ollama server is up ✅


### 1.4 Download the model and test it
`MODEL` is the one place where the model name lives. Every later cell reads it.

**Which model:** this notebook now uses **`llama3.1:8b`** (about 5 GB to download). On your 3B runs, the answers sometimes used the model's own knowledge instead of the PDF, and the quizzes had weak answer keys. An 8B model follows "use only the context" better. It fits easily on the free T4 GPU, but it is slow on a CPU runtime. **On CPU, change it back to `llama3.2:3b`.**

**Check:** a short greeting is printed, the GPU line shows your GPU, and `ollama ps` says `100% GPU`.

In [ ]:
MODEL = "llama3.1:8b"      # on a CPU runtime use "llama3.2:3b" (smaller and faster)

!ollama pull {MODEL} 2>&1 | tail -n 2

import ollama
reply = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "Say hello in one short sentence."}],
)
print(reply["message"]["content"])

print("\n--- Which processor? ---")
!nvidia-smi -L
!ollama ps

writing manifest 
success 
Hello!

--- Which processor? ---
GPU 0: Tesla T4 (UUID: GPU-fc2e5c3e-05b0-5c11-d946-50b08dbaf045)
NAME           ID              SIZE      PROCESSOR    CONTEXT    RUNNER      UNTIL              
llama3.1:8b    46e0c10c039e    5.3 GB    100% GPU     4096       llamacpp    4 minutes from now    


### 1.5 Load the embedding model
It turns text into 384 numbers that capture its meaning. Similar meaning gives similar numbers.
**Check:** `Embedding length: 384`. Pinecone must be created with this same number later.

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
vec = embedder.encode("What is machine learning?")
print("Embedding length:", len(vec))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding length: 384


### 1.6 Connect to Pinecone
It first tries a Colab Secret named `PINECONE_API_KEY` (🔑 icon in the left sidebar, with Notebook access on). If there is none, it asks you to paste the key in a hidden box. Never paste a key into the code itself.
**Check:** `Connected to Pinecone ✅` and a list of your indexes.

In [ ]:
from getpass import getpass
from pinecone import Pinecone

try:
    from google.colab import userdata
    api_key = userdata.get("PINECONE_API_KEY")
except Exception:
    api_key = getpass("Paste your Pinecone API key: ")

pc = Pinecone(api_key=api_key)
print("Connected to Pinecone ✅")
print("Existing indexes:", pc.list_indexes().names())

Paste your Pinecone API key: ··········
Connected to Pinecone ✅
Existing indexes: ['pdf-rag', 'study-buddy']


# Part 2 — Load and clean the PDF

### 2.1 Upload your PDF
**Check:** it prints the file name and size.

In [ ]:
from google.colab import files
import os

uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print("Uploaded:", pdf_path)
print("Size (KB):", round(os.path.getsize(pdf_path) / 1024, 1))

Saving Retrieval-augmented generation - Wikipedia.pdf to Retrieval-augmented generation - Wikipedia.pdf
Uploaded: Retrieval-augmented generation - Wikipedia.pdf
Size (KB): 253.7


### 2.2 Read, clean, and cut off the references
What happens here, step by step:
1. **Read** every page and keep its page number (we need it to cite sources).
2. **Remove repeated lines** that appear on at least half of the pages (headers and footers).
3. **Clean** each page: fix special joined letters (ﬁ → fi), remove citation marks like `[12]`, join words split across lines, squeeze spaces.
4. **Cut the References section.** It is a list of paper titles and links. It confuses search, so everything from the heading `References 1.` onwards is dropped.
5. **Mark section headings** (short lines like "Chunking" or "Hybrid search"). The chunker starts a new chunk at every heading. Before, the heading "Chunking" stayed glued to the end of the previous section, so a summary of chunking also talked about the Retro model. The marker is the character `¶`, and `plain()` removes it where it is not needed. If a PDF has no clear headings, set `DETECT_HEADINGS = False`.

**Correction:** the old version removed reference chunks *after* chunking, and it also threw away a real paragraph on page 6 that sat next to the references. Cutting at the heading keeps that paragraph.
**Check:** `Pages kept` should be 6 for the Wikipedia RAG PDF, the headings list should be real section titles (13 for this PDF), and the preview should look like normal sentences.

In [ ]:
from pypdf import PdfReader
from collections import Counter
import re, unicodedata

reader = PdfReader(pdf_path)
raw_pages = [{"page": i + 1, "text": p.extract_text() or ""} for i, p in enumerate(reader.pages)]
n_pages = len(raw_pages)

def norm_line(line):
    return re.sub(r"\d+", "#", line.strip().lower())      # "Page 2/8" -> "page #/#"

line_counts = Counter()
for p in raw_pages:
    for line in {norm_line(l) for l in p["text"].split("\n") if l.strip()}:
        line_counts[line] += 1

repeated = set()
if n_pages >= 4:
    repeated = {line for line, c in line_counts.items()
                if c >= max(3, int(0.5 * n_pages)) and len(line) < 100}

PAGE_NUM = re.compile(r"(page\s*)?\d+(\s*(/|of)\s*\d+)?")

DETECT_HEADINGS = True

def is_heading(line):
    """A short line, 1-5 words, starts with a capital, only letters/digits/hyphens: "Chunking", "Hybrid search"."""
    t = line.strip()
    return (DETECT_HEADINGS and 1 <= len(t.split()) <= 5 and t[0].isupper()
            and re.fullmatch(r"[A-Za-z0-9+\- ]+", t) is not None and t.lower() != "references")

def plain(text):
    return text.replace("¶", "")

def clean_text(text):
    text = unicodedata.normalize("NFKC", text)                       # ﬁ -> fi
    text = re.sub(r"\[\d{1,3}\]", "", text)                          # [1] [23]
    text = re.sub(r"\[(citation needed|edit)\]", "", text, flags=re.I)
    text = re.sub(r"-\n(\w)", r"\1", text)                           # join split words
    text = re.sub(r"\n+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

pages = []
for p in raw_pages:
    kept = [l for l in p["text"].split("\n")
            if norm_line(l) not in repeated and not PAGE_NUM.fullmatch(l.strip().lower())]
    kept = [f"¶{l.strip()}¶" if l.strip() and is_heading(l) else l for l in kept]   # mark headings
    text = clean_text("\n".join(kept))
    if len(text) > 30:
        pages.append({"page": p["page"], "text": text})

# Cut everything from the References heading onwards
REF_START = re.compile(r"\bReferences\s+1\.\s")
cut_pages, refs_found = [], False
for p in pages:
    if refs_found:
        break
    m = REF_START.search(p["text"])
    if m:
        refs_found = True
        before = p["text"][:m.start()].strip()
        if len(before) > 30:
            cut_pages.append({"page": p["page"], "text": before})
    else:
        cut_pages.append(p)
pages = cut_pages

print("PDF pages:", n_pages, "| pages kept:", len(pages), "| references cut:", refs_found)
print("Repeated header/footer lines removed:", sorted(repeated) or "none")
print("Headings found:", [h for p in pages for h in re.findall(r"¶([^¶]+)¶", p["text"])])
print("\nPreview of page 1:\n", plain(pages[0]["text"])[:400])

PDF pages: 8 | pages kept: 6 | references cut: True
Repeated header/footer lines removed: none
Headings found: ['Retrieval-augmented generation', 'RAG and LLM limitations', 'Process', 'RAG key stages', 'Applications', 'Improvements', 'Encoder', 'Retriever-centric methods', 'Language model', 'Chunking', 'Hybrid search', 'Challenges', 'RAG poisoning']

Preview of page 1:
 Retrieval-augmented generation Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information from external data sources. With RAG, LLMs first refer to a specified set of documents, then respond to user queries. These documents supplement information from the LLM's pre-existing training data. This allows LLMs to use domain-


### 2.3 Check the cleaned result
**Check:** totals are above 0, the preview reads as smooth sentences, and nothing like `[1]` or `doi:` is left.

In [ ]:
total_words = sum(len(p["text"].split()) for p in pages)
print("Total words (approx):", total_words)
print("Leftover [n] marks:", sum(len(re.findall(r"\[\d+\]", p["text"])) for p in pages))
print("Leftover 'doi:' mentions:", sum(p["text"].lower().count("doi:") for p in pages))
print("\nLast 200 characters of the last page:\n", plain(pages[-1]["text"])[-200:])

Total words (approx): 1631
Leftover [n] marks: 0
Leftover 'doi:' mentions: 0

Last 200 characters of the last page:
 ountry (hasbara) after support plummeted due to its genocidal conduct during the Gaza war. Clock Tower X used RAG poisoning to flood the information space with sites and content sympathetic to Israel.


# Part 3 — Split the text into chunks

### 3.1 The chunking function
**Why chunk?** The model can't read the whole PDF for every question. We cut the text into pieces of about 150 words. Search later finds the few pieces that match the question.

How it works:
- Each page is split into **sentences**, and whole sentences are packed into a chunk until it reaches about 150 words. So a chunk never stops in the middle of a sentence.
- A **section heading** (marked in Part 2) starts a new chunk, so one chunk does not mix two topics. The heading is kept as the first words of its chunk.
- The next chunk **repeats the last ~30 words** of the previous one (overlap), so an idea at the edge is not lost.
- Chunks never cross a page boundary, so every chunk has an exact page number for citing.
- Tiny leftover scraps (under 30 words) are skipped.

In [ ]:
CHUNK_SIZE = 150    # target words per chunk
OVERLAP = 30        # words repeated from the previous chunk
MIN_WORDS = 30

SENT_SPLIT = re.compile(r'(?<=[.!?])\s+(?=[A-Z0-9"“(])')
HEAD_RE = re.compile(r"¶([^¶]+)¶")

def split_sentences(text):
    return [s.strip() for s in SENT_SPLIT.split(text) if s.strip()]

def words_in(sentences):
    return sum(len(x.split()) for x in sentences)

def split_into_chunks(pages, chunk_size=CHUNK_SIZE, overlap=OVERLAP):
    chunks = []
    for p in pages:
        parts = HEAD_RE.split(p["text"])                  # [text, heading, text, heading, ...]
        items = []
        for i, part in enumerate(parts):
            if i % 2 == 1:
                items.append(("heading", part.strip() + "."))
            else:
                items += [("sentence", s) for s in split_sentences(part)]

        start_count, current = len(chunks), []
        for kind, s in items:
            if kind == "heading" and words_in(current) >= MIN_WORDS:
                chunks.append({"id": f"chunk-{len(chunks)}", "page": p["page"], "text": " ".join(current)})
                current = []                               # a new section starts a new chunk
            elif kind == "sentence" and current and words_in(current) + len(s.split()) > chunk_size:
                chunks.append({"id": f"chunk-{len(chunks)}", "page": p["page"], "text": " ".join(current)})
                keep, kept = [], 0                         # carry the last sentences over as overlap
                for prev in reversed(current):
                    w = len(prev.split())
                    if kept + w > overlap:
                        break
                    keep.insert(0, prev)
                    kept += w
                current = keep
            current.append(s)
        if current:
            text = " ".join(current)
            if len(text.split()) >= MIN_WORDS or len(chunks) == start_count:
                chunks.append({"id": f"chunk-{len(chunks)}", "page": p["page"], "text": text})
    return chunks

### 3.2 Create the chunks and check their quality
There is also a safety filter. If any chunk still looks like a reference list (many years plus words like *doi* or *arXiv*), it is dropped.
**Check:**
- About 14-20 chunks for the Wikipedia RAG PDF (headings add a few; other PDFs will differ).
- Words per chunk: max 150, min 30 or more.
- Almost all chunks end at a sentence end.
- Chunk text must never contain the heading marker `¶`.
- Every page you kept appears in "Chunks per page".
- `Reference-like chunks removed: 0`.

In [ ]:
REF_WORDS = re.compile(r"\b(doi|arxiv|isbn|s2cid|pmid|retrieved|archived)\b", re.I)
YEAR = re.compile(r"\b(19|20)\d{2}\b")

def looks_like_reference(text):
    return len(REF_WORDS.findall(text)) + len(YEAR.findall(text)) >= 8

chunks = split_into_chunks(pages)
before = len(chunks)
chunks = [c for c in chunks if not looks_like_reference(c["text"])]

lengths = [len(c["text"].split()) for c in chunks]
ends_ok = sum(c["text"].rstrip()[-1] in '.!?"”)' for c in chunks)

print("Total chunks:", len(chunks))
print("Reference-like chunks removed:", before - len(chunks))
print(f"Words per chunk  min/avg/max: {min(lengths)}/{round(sum(lengths)/len(lengths))}/{max(lengths)}")
print(f"Chunks ending at a sentence end: {ends_ok}/{len(chunks)}")
print("Chunks per page:", dict(sorted(Counter(c['page'] for c in chunks).items())))
print("Chunks containing the marker ¶:", sum("¶" in c["text"] for c in chunks))
print("\n--- Chunk 0 (page", chunks[0]["page"], ") ---\n", chunks[0]["text"][:300], "...")

Total chunks: 19
Reference-like chunks removed: 0
Words per chunk  min/avg/max: 31/90/148
Chunks ending at a sentence end: 18/19
Chunks per page: {1: 4, 2: 3, 3: 4, 4: 3, 5: 4, 6: 1}
Chunks containing the marker ¶: 0

--- Chunk 0 (page 1 ) ---
 Retrieval-augmented generation. Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information from external data sources. With RAG, LLMs first refer to a specified set of documents, then respond to user queries. These docume ...


# Part 4 — Store the chunks in Pinecone

### 4.1 Create the index
An index is like a table for vectors. `dimension=384` must match the embedding model, and `cosine` compares meaning by angle.
**Correction:** it now also checks that an existing index has dimension 384. If you reuse an old index made for another model, you get a clear error instead of a confusing failure later.
**Check:** `Index ready ✅`.

In [ ]:
from pinecone import ServerlessSpec
import time

INDEX_NAME = "study-buddy"

if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        time.sleep(1)

dim = pc.describe_index(INDEX_NAME).dimension
assert dim == 384, f"Index '{INDEX_NAME}' has dimension {dim}, but the embedder makes 384. Use another INDEX_NAME."

index = pc.Index(INDEX_NAME)
print("Index ready ✅:", INDEX_NAME)

Index ready ✅: study-buddy


### 4.2 Turn every chunk into an embedding
**Check:** the number of embeddings equals your chunk count, and each has length 384.

In [ ]:
texts = [c["text"] for c in chunks]
embeddings = embedder.encode(texts, show_progress_bar=False)

print("Number of embeddings:", len(embeddings))
print("Each embedding length:", len(embeddings[0]))

Number of embeddings: 19
Each embedding length: 384


### 4.3 Upload to Pinecone (clearing old data first)
- The old vectors are deleted first, so answers can never mix two different PDFs.
- Each record stores the embedding **plus the page number and the original text** as metadata, so search can return readable text.
- Uploads go in batches of 50.
- **Correction:** Pinecone's count updates a few seconds late. Instead of a blind `sleep(10)`, the cell now checks the count until it matches (up to about a minute).

**Check:** `Vectors in Pinecone` equals `chunks`, with ✅.

In [ ]:
try:
    index.delete(delete_all=True)
    time.sleep(3)
    print("Old vectors cleared")
except Exception:
    print("Index was already empty")

vectors = [{"id": c["id"], "values": emb.tolist(),
            "metadata": {"page": c["page"], "text": c["text"]}}
           for c, emb in zip(chunks, embeddings)]

BATCH = 50
for i in range(0, len(vectors), BATCH):
    index.upsert(vectors=vectors[i:i + BATCH])

n = 0
for _ in range(30):
    n = index.describe_index_stats()["total_vector_count"]
    if n == len(vectors):
        break
    time.sleep(2)

print("Vectors in Pinecone:", n, "| chunks:", len(chunks), "✅" if n == len(vectors) else "⚠️ counts differ, wait and re-run this cell")

Old vectors cleared
Vectors in Pinecone: 19 | chunks: 19 ✅


# Part 5 — Search with reranking

### 5.1 The search function
Search has two stages:
1. **Fast stage:** Pinecone returns the 12 chunks closest in meaning (`cosine`).
2. **Careful stage:** a **reranker** reads the question and each chunk *together* and gives a better score (`rerank`). We keep the best 4.

Why both? The fast stage sometimes ranks a chunk that only *mentions* the topic above the one that *explains* it. The reranker fixes that.

**Correction (important):** the old version stored the cosine number in `score`, so the "I don't know" guard compared the wrong number. Now `score` holds the **rerank** score everywhere, and the original is kept as `cosine`.

Two settings:
- `REL_MIN`: a chunk must score above this to be used as context. This stops irrelevant chunks (and irrelevant page numbers) from reaching the answer.
- `MIN_SCORE`: if even the best chunk scores below this, the bot says "I don't know". It is set from real data in Part 7.

**Check:** the page 1 definition ranks first for "What is retrieval-augmented generation?", with a clearly higher rerank score than the others.

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

MIN_SCORE = 0.0   # "I don't know" cutoff, calibrated in Part 7
REL_MIN = 0.0     # a chunk needs a rerank score above this to be used as context

def search(query, top_k=4, fetch_k=12):
    q_emb = embedder.encode(query).tolist()
    result = index.query(vector=q_emb, top_k=fetch_k, include_metadata=True)
    cands = [{"cosine": m["score"], "page": m["metadata"]["page"], "text": m["metadata"]["text"]}
             for m in result["matches"]]
    if not cands:
        return []
    scores = reranker.predict([(query, c["text"]) for c in cands])
    for c, s in zip(cands, scores):
        c["rerank"] = float(s)
        c["score"] = float(s)          # the guards read "score"
    cands.sort(key=lambda c: c["rerank"], reverse=True)
    return cands[:top_k]

def relevant_hits(query, top_k=4):
    """Only the chunks that really match; always at least the best one."""
    hits = search(query, top_k=top_k)
    return [h for h in hits if h["rerank"] >= REL_MIN] or hits[:1]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [ ]:
for q in ["What is retrieval-augmented generation?", "How does RAG reduce hallucinations?", "What is the capital of France?"]:
    print("Q:", q)
    for h in search(q):
        print(f"  rerank {h['rerank']:6.2f} | cosine {h['cosine']:.3f} | page {h['page']} | {h['text'][:80]}...")
    print()

Q: What is retrieval-augmented generation?
  rerank  10.18 | cosine 0.516 | page 1 | Retrieval-augmented generation. Retrieval-augmented generation (RAG) is a techni...
  rerank   9.26 | cosine 0.629 | page 3 | Applications. Retrieval-augmented generation is used in applications where gener...
  rerank   9.09 | cosine 0.617 | page 1 | This provides greater transparency, as users can cross-check retrieved content t...
  rerank   8.66 | cosine 0.685 | page 2 | Process. Retrieval-augmented generation (RAG) enhances large language models (LL...

Q: How does RAG reduce hallucinations?
  rerank   7.60 | cosine 0.308 | page 1 | RAG improves LLMs by incorporating information retrieval before generating respo...
  rerank   6.89 | cosine 0.550 | page 5 | Challenges. RAG does not prevent hallucinations in LLMs. According to Ars Techni...
  rerank  -4.41 | cosine 0.142 | page 1 | RAG and LLM limitations. LLMs can provide incorrect information. For example, wh...
  rerank  -5.03 | cosine 0.420 | pa

# Part 6 — The answer function (the RAG loop)

### 6.1 Answer a question
The whole RAG loop in one function:
1. **Retrieve:** find the best chunks.
2. **Guard:** if even the best chunk is a poor match, answer "I don't know" without asking the model. This stops made-up answers.
3. **Augment:** put only the *relevant* chunks (with page numbers) into the prompt.
4. **Generate:** the model answers using only that context.
5. **Return:** the answer, the pages actually used, and the top score.

**The grounding check and its fallback.** A small model sometimes answers from its own memory instead of the PDF (for example, it explained "vector database" although the PDF never defines it). `grounding()` measures what share of the answer's key words appear in the context. Real answers score 0.7-1.0, answers from outside knowledge score about 0.3.
Below `GROUND_MIN`, the answer is **replaced** (this is the fix):
1. The model is asked to copy the sentences that answer the question, word for word.
2. `evidence_in_context()` checks that the copied text really is in the PDF chunks. If it is, the bot shows `Closest text in the PDF: “...”`, with the right page.
3. If the model finds no such sentence, the bot says "I don't know. The PDF doesn't explain this clearly."

So the bot can no longer present the model's own knowledge as if it came from the PDF. The result also reports `raw_grounding` (before the fallback) and `fallback`, which the evaluation uses to measure how often the model needed correcting.

Corrections here: the prompt now forbids side notes and invented acronym expansions, it says to answer the question first and mention caveats last, only relevant chunks are used, and if the model itself says "I don't know", no pages are listed.

`temperature` controls randomness. Normal chat uses 0.2. The evaluation uses 0, so a re-run gives (almost) the same result and a change in score means something.

In [ ]:
IDK = "I don't know. This isn't covered in the PDF."

# --- "Is the answer really from the PDF?" check -------------------------------
STOPWORDS = set("a an the of to in on for and or is are was were be been being it its this that these those as by with from at which who whom whose their there than then also can may might not does did has have had into over such more most other some any each both about between during".split())

def _stem(w):
    return w[:6]                      # crude word stem: "reproducibility" and "reproducible" match

def content_stems(s):
    return {_stem(w) for w in re.findall(r"[a-z0-9]+", s.lower()) if len(w) > 3 and w not in STOPWORDS}

def grounding(text, source):
    """Share of the answer's key words that also appear in the source text (0 to 1)."""
    a = content_stems(text)
    return len(a & content_stems(source)) / len(a) if a else 1.0

GROUND_MIN = 0.45   # below this, the answer probably uses the model's own knowledge

# --- "Is this sentence really in the text?" check (used by answers, quizzes and summaries) ---
import difflib

def _norm(s):
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", s.lower())).strip()

def evidence_in_context(evidence, context, min_ratio=0.9):
    """True if (almost) all of the evidence sentence appears in the context."""
    ev, ctx = _norm(evidence), _norm(context)
    if len(ev.split()) < 4:
        return False
    if ev in ctx:
        return True
    m = difflib.SequenceMatcher(None, ev, ctx, autojunk=False).find_longest_match(0, len(ev), 0, len(ctx))
    return m.size / len(ev) >= min_ratio

IDK_SOFT = "I don't know. The PDF doesn't explain this clearly."

def extract_support(question, context):
    """Ask the model to COPY the sentences that answer the question. Returns them only if they really are in the context."""
    prompt = f"""Copy the one or two sentences from the context that best answer the question.
Copy them word for word. Do not add anything. Do not change any word.
If no sentence answers the question, reply exactly: I don't know.

Context:
{context}

Question: {question}"""
    try:
        reply = ollama.chat(model=MODEL, messages=[{"role": "user", "content": prompt}],
                            options={"temperature": 0, "num_ctx": 4096})
        out = re.sub(r"\[Page \d+\]\s*", "", reply["message"]["content"]).strip().strip('"“”')
    except Exception:
        return None
    if not out or says_unknown(out) or not evidence_in_context(out, context, min_ratio=0.85):
        return None
    return out

def says_unknown(text):
    t = text.lower().replace("’", "'")
    return "don't know" in t or "do not know" in t

def answer(question, top_k=4, hits=None, temperature=0.2):
    hits = hits if hits is not None else search(question, top_k)

    if not hits or hits[0]["score"] < MIN_SCORE:
        return {"answer": IDK, "pages": [], "top_score": hits[0]["score"] if hits else 0.0,
                "grounding": 1.0, "raw_grounding": 1.0, "fallback": None}

    used = [h for h in hits if h["rerank"] >= REL_MIN] or hits[:1]
    context = "\n\n".join(f"[Page {h['page']}] {h['text']}" for h in used)

    system_msg = (
        "You are a study assistant. Answer ONLY using the context provided. "
        "Stay close to the wording of the context. "
        "Answer the question that was asked first. If the context also mentions "
        "a limitation or caveat, add it in one final sentence. "
        "Match the exact terms of the question (for example a named feature); "
        "ignore passages that only share a similar word. "
        "If the context does not contain the answer, reply exactly: I don't know. "
        "Do not add notes, assumptions or comments about the context. "
        "Do not expand acronyms unless the context does. "
        "Keep the answer clear and short (2-4 sentences)."
    )
    user_msg = f"Context:\n{context}\n\nQuestion: {question}"

    reply = ollama.chat(
        model=MODEL,
        messages=[{"role": "system", "content": system_msg},
                  {"role": "user", "content": user_msg}],
        options={"temperature": temperature, "num_ctx": 4096},
    )
    text = reply["message"]["content"].strip()
    unknown = says_unknown(text)
    raw_ground = 1.0 if unknown else grounding(text, context)

    # Grounding fallback: the answer uses words that are not in the PDF text.
    # Replace it with a verbatim quote from the PDF, or an honest "I don't know".
    fallback, quote = None, None
    if not unknown and raw_ground < GROUND_MIN:
        quote = extract_support(question, context)
        if quote:
            text, fallback = f"Closest text in the PDF: “{quote}”", "quote"
        else:
            text, fallback = IDK_SOFT, "unknown"

    if says_unknown(text):
        pages_used = []
    elif quote:
        pages_used = sorted({h["page"] for h in used if evidence_in_context(quote, h["text"], 0.85)}) \
                     or sorted({h["page"] for h in used})
    else:
        pages_used = sorted({h["page"] for h in used})
    return {"answer": text, "pages": pages_used, "top_score": hits[0]["score"],
            "grounding": 1.0 if fallback else raw_ground, "raw_grounding": raw_ground, "fallback": fallback}

### 6.2 Two quick tests
**Check:** the first answer is correct, short, and cites pages. The second says "I don't know" with no pages.

In [ ]:
for q in ["What is retrieval-augmented generation?", "Who won the football World Cup in 2018?"]:
    r = answer(q)
    print("Q:", q)
    print("Answer:", r["answer"])
    print("Pages used:", r["pages"], "| top score:", round(r["top_score"], 2), "\n")

Q: What is retrieval-augmented generation?
Answer: Retrieval-augmented generation (RAG) is a technique that enables large language models to retrieve and incorporate new information from external data sources. It allows LLMs to use domain-specific and/or updated information that is not available in the training data.
Pages used: [1, 2, 3] | top score: 10.18 

Q: Who won the football World Cup in 2018?
Answer: I don't know. This isn't covered in the PDF.
Pages used: [] | top score: -10.82 



### 6.3 Peek at what the bot saw (debug helper)
When an answer looks wrong, run `peek("your question")`. It shows every chunk found, its rerank score, and whether it was used as context, then the answer and its grounding score. This tells you in seconds whether the problem is **search** (the right chunk is missing or ranked low) or the **model** (it had the right chunk and still answered badly).

In [ ]:
def peek(question):
    hits = search(question)
    print("Q:", question, "\n")
    for h in hits:
        used = "USED   " if h["rerank"] >= REL_MIN else "skipped"
        print(f"  {used} rerank {h['rerank']:6.2f} | page {h['page']} | {h['text'][:90]}...")
    r = answer(question, hits=hits)
    print("\nAnswer:", r["answer"])
    flag = "  ⚠️ LOW, fallback used: " + r["fallback"] if r["fallback"] else "  ✅"
    print(f"Pages: {r['pages']} | grounding of the model's own answer: {r['raw_grounding']:.2f}{flag}")

peek("What is a vector database?")

Q: What is a vector database? 

  USED    rerank   4.40 | page 5 | Hybrid search. Sometimes vector database searches (aka semantic search technique) can miss...
  skipped rerank  -0.42 | page 4 | Retriever-centric methods. These methods aim to enhance the quality of document retrieval ...
  skipped rerank  -1.11 | page 3 | Improvements. Improvements to the basic process above can be applied at different stages i...
  skipped rerank  -1.69 | page 2 | RAG key stages. Overview of RAG process, combining external documents and user input into ...

Answer: I don't know.
Pages: [] | grounding of the model's own answer: 1.00  ✅


# Part 7 — Evaluation: measure the real accuracy

### 7.1 The test set (46 questions)
38 questions the PDF answers and 8 it should refuse. The last 10 are harder: reworded or two-part questions, and 2 **traps** (they sound like the PDF's topic, but the PDF never says the answer, so the right output is "I don't know"). Each in-PDF question lists the correct page(s) and keyword groups: a correct answer must contain at least one keyword from **every** group. A page list like `[1, 2]` means the answer sits near a page boundary, so either page counts.
This set is written for the Wikipedia RAG PDF. For another PDF, write new questions.

In [ ]:
# (question, correct pages, keyword groups). Empty pages = off-topic: the right answer is "I don't know".
eval_set = [
    ("What is retrieval-augmented generation?", [1], [["retriev"], ["external", "data source", "new information"]]),
    ("When was RAG first proposed?", [1], [["2020"]]),
    ("How does RAG help reduce AI hallucinations?", [1], [["retriev", "search", "document", "look-up"]]),
    ("Why does RAG save money compared to retraining LLMs?", [1], [["retrain"], ["cost", "computational", "financial", "sav"]]),
    ("How does RAG make answers more transparent?", [1], [["source", "cite", "verify", "cross-check"]]),
    ("What mistake did Google's Bard make in its demonstration?", [1], [["james webb", "telescope"]]),
    ("How much did Google's stock value decline after the Bard error?", [1], [["100 billion", "$100"]]),
    ("Why did the model wrongly say the US had a Muslim president?", [1, 2], [["title", "chapter", "context", "misinterpret", "misunderstood"]]),
    ("What is prompt stuffing?", [2], [["context", "add"], ["prompt", "input"]]),
    ("What does the generative phase do, according to IBM?", [2], [["augmented", "prompt"], ["training", "internal"]]),
    ("In what form is data converted before it is stored in a vector database?", [2], [["embedding"]]),
    ("What does the document retriever do when it gets a user query?", [2], [["relevant"], ["document"]]),
    ("What abilities do augmentation modules add to newer RAG implementations?", [3], [["expand", "memory", "self-improvement", "multiple domains"]]),
    ("What extra steps can some models use to improve output?", [3], [["re-rank", "rerank", "context selection", "fine-tuning"]]),
    ("How has RAG been studied in healthcare?", [3], [["medical", "knowledge", "ground"]]),
    ("What is the difference between sparse and dense vectors?", [3], [["sparse"], ["dense"], ["meaning", "word", "zero"]]),
    ("Why use approximate nearest neighbor search instead of KNN?", [3], [["efficien"]]),
    ("What are hybrid vector approaches?", [3], [["dense"], ["sparse"]]),
    ("What do late interactions do?", [3], [["compare", "precis", "ranking"]]),
    ("What is SPLADE combined with to improve search accuracy?", [4], [["query expansion"]]),
    ("What is the Inverse Cloze Task?", [4], [["pre-train", "retriev", "mask"]]),
    ("What does supervised retriever optimization do?", [4], [["align", "likelihood", "perplexity", "kl"]]),
    ("What advantage does the Retro language model have?", [4], [["25", "smaller"]]),
    ("What is Retro++?", [4], [["reproducib"], ["in-context", "retro"]]),
    ("What is chunking in RAG?", [5], [["break", "split", "divid", "strateg"]]),
    ("What is hybrid search?", [5], [["traditional", "full text", "text search", "keyword"], ["vector", "semantic"]]),
    ("Does RAG completely prevent hallucinations?", [5], [["not"], ["hallucinat"]]),
    ("Why might an LLM answer even when it lacks enough information?", [5], [["uncertain", "limitation", "assess", "specific training", "own knowledge"]]),
    ("What is RAG poisoning?", [5, 6], [["mislead", "incorrect", "conflict", "misinterpret"]]),
    ("What can happen when RAG faces conflicting information?", [5], [["struggle", "determine", "merge", "outdated", "accurate"]]),
    ("Who won the football World Cup in 2018?", [], []),
    ("What is the capital of France?", [], []),
    ("How do I bake a chocolate cake?", [], []),
    ("What is the boiling point of water?", [], []),
    ("Who wrote the play Hamlet?", [], []),
    ("What is the best programming language for web development?", [], []),
    # Harder questions: reworded, two-part, or about things the PDF never says
    ("Why don't companies have to retrain the model as often when they use RAG?", [1, 2], [["retrain"], ["new", "updat", "external", "knowledge base"]]),
    ("What are the limitations of RAG, and what is RAG poisoning?", [5], [["retrain", "hallucinat", "uncertain", "limitation"], ["mislead", "incorrect", "conflict", "misinterpret"]]),
    ("What went wrong when a model treated a book chapter title as a fact?", [1, 2], [["muslim", "title", "chapter", "obama", "false", "misinterpret"]]),
    ("How do sparse and dense vectors differ, and what is a hybrid approach?", [3], [["sparse"], ["dense"], ["combine", "hybrid"]]),
    ("What happens to the model's input when prompt stuffing is used?", [2], [["context", "additional", "add"], ["input", "prompt"]]),
    ("Which approach lets a network 25 times smaller reach similar perplexity?", [4], [["retro"]]),
    ("Can RAG be used on structured data such as knowledge graphs?", [2], [["structured", "knowledge graph"]]),
    ("According to IBM, what ability may an LLM lack when information is missing?", [5], [["assess", "own knowledge", "limitation", "uncertain"]]),
    # Traps: close to the topic, but the PDF does not say. The right answer is "I don't know".
    ("What is the best vector database to use with RAG?", [], []),
    ("How many tokens can a RAG system process at once?", [], []),
]

# The two traps are NOT used to set the cutoff in 7.2, so they stay a fair, unseen test.
TRAPS = {"What is the best vector database to use with RAG?",
         "How many tokens can a RAG system process at once?"}
print("In-PDF questions:", sum(1 for e in eval_set if e[1]), "| off-topic:", sum(1 for e in eval_set if not e[1]))

In-PDF questions: 38 | off-topic: 8


### 7.2 Set the "I don't know" cutoff from real data
**Correction:** the old cutoff (`0.25`) was a guess. This cell measures the best rerank score of every in-PDF question and of every clearly off-topic question, then puts `MIN_SCORE` halfway between the two groups.
The 2 trap questions are left out on purpose and only shown at the end, so they remain a fair test of the guard.
**Check:** the in-PDF lowest score should be clearly above the off-topic highest score, with ✅. If the scores overlap, it keeps the old value and warns you. The trap scores should be below `MIN_SCORE`. If one is above it, the guard cannot catch that trap (that is a real finding for 7.4).

In [ ]:
def calibrate_min_score():
    global MIN_SCORE
    known = [search(q)[0]["score"] for q, pg, _ in eval_set if pg]
    unknown = [search(q)[0]["score"] for q, pg, _ in eval_set if not pg and q not in TRAPS]
    lo, hi = min(known), max(unknown)
    print("In-PDF questions, lowest top score:    ", round(lo, 2))
    print("Off-topic questions, highest top score:", round(hi, 2))
    if lo > hi:
        MIN_SCORE = round((lo + hi) / 2, 2)
        print("✅ New MIN_SCORE:", MIN_SCORE)
    else:
        print("⚠️ The scores overlap. Keeping MIN_SCORE =", MIN_SCORE, "(look at the weakest questions in 7.4)")
    for q in TRAPS:
        sc = search(q)[0]["score"]
        print(f"Trap score {sc:6.2f} ({'below' if sc < MIN_SCORE else 'ABOVE'} MIN_SCORE) | {q}")

calibrate_min_score()

In-PDF questions, lowest top score:     -2.64
Off-topic questions, highest top score: -8.55
✅ New MIN_SCORE: -5.6
Trap score  -2.75 (ABOVE MIN_SCORE) | How many tokens can a RAG system process at once?
Trap score   3.08 (ABOVE MIN_SCORE) | What is the best vector database to use with RAG?


### 7.3 Run the evaluation
For every question it measures:
1. **Retrieval hit rate:** is a chunk from the correct page among the top 4?
2. **Answer accuracy:** is the answer not "I don't know", and does it contain a keyword from every group?
3. **Refusal rate:** do off-topic questions get "I don't know"?

Results are stored under a label in `EVAL_LOG`, so you can run again after a change and compare. Use a **new label for every run** (the last line of the cell uses the model name), or the old result is overwritten. To compare models, change `MODEL` in 1.4, re-run the cells from 1.4 onward, then run this part again: `compare_runs()` shows the runs side by side, including the number of low-grounding answers and the time.

The test runs the model at temperature 0, so repeated runs give (almost) the same score.

**Quick mode** (for a slow CPU runtime): `run_eval("my label", quick=True)` tests only every 3rd question (about 16). Use it for fast checks, and run the full set for your final numbers.
A full run takes a few minutes on GPU and roughly 10-15 minutes on CPU.
**Remember:** the keyword check is a rough proxy. Always read the failures in 7.4 yourself.

In [ ]:
EVAL_LOG = {}

def run_eval(label="run 1", quick=False):
    start = time.time()
    rows = []
    items = eval_set[::3] if quick else eval_set
    for q, pages_ok, must in items:
        hits = search(q)
        res = answer(q, hits=hits, temperature=0)
        ans, g, fb = res["answer"], res["raw_grounding"], res["fallback"]
        low = ans.lower().replace("’", "'")
        unknown = says_unknown(ans)
        if pages_ok:                                   # the PDF covers this
            retr_ok = any(h["page"] in pages_ok for h in hits)
            ans_ok = (not unknown) and all(any(k in low for k in g) for g in must)
        else:                                          # off-topic
            retr_ok = None
            ans_ok = unknown
        rows.append({"q": q, "pages_ok": pages_ok, "got_pages": [h["page"] for h in hits],
                     "retr_ok": retr_ok, "ans_ok": ans_ok, "answer": ans, "grounding": g, "fallback": fb})

    inpdf = [r for r in rows if r["pages_ok"]]
    off = [r for r in rows if not r["pages_ok"]]
    summary = {
        "retrieval_hit": sum(r["retr_ok"] for r in inpdf) / len(inpdf),
        "answer_acc":    sum(r["ans_ok"] for r in inpdf) / len(inpdf),
        "refusal_rate":  sum(r["ans_ok"] for r in off) / len(off),
        "overall":       sum(r["ans_ok"] for r in rows) / len(rows),
        "low_grounded":  sum(r["grounding"] < GROUND_MIN for r in rows),
        "fallbacks":     sum(r["fallback"] is not None for r in rows),
        "seconds":       round(time.time() - start),
    }
    EVAL_LOG[label] = {"summary": summary, "rows": rows}
    print(f"[{label}]  ({len(rows)} questions{', quick mode' if quick else ''})")
    print(f"  Retrieval hit rate (right page in top 4): {summary['retrieval_hit']:.0%}")
    print(f"  Answer accuracy on PDF questions:         {summary['answer_acc']:.0%}")
    print(f"  Off-topic correctly refused:              {summary['refusal_rate']:.0%}")
    print(f"  Overall:                                  {summary['overall']:.0%}")
    print(f"  Model answers with LOW grounding (outside knowledge?): {summary['low_grounded']}")
    print(f"  Answers replaced by the grounding fallback:           {summary['fallbacks']}")
    print(f"  Time: {summary['seconds']} s   | model: {MODEL}")
    return summary

run_eval(f"{MODEL}, temp 0")

[llama3.1:8b, temp 0]  (46 questions)
  Retrieval hit rate (right page in top 4): 100%
  Answer accuracy on PDF questions:         97%
  Off-topic correctly refused:              100%
  Overall:                                  98%
  Model answers with LOW grounding (outside knowledge?): 0
  Answers replaced by the grounding fallback:           0
  Time: 59 s   | model: llama3.1:8b


{'retrieval_hit': 1.0,
 'answer_acc': 0.9736842105263158,
 'refusal_rate': 1.0,
 'overall': 0.9782608695652174,
 'low_grounded': 0,
 'fallbacks': 0,
 'seconds': 59}

### 7.4 See what failed
How to read a failure:
- **RETRIEVAL:** the right page was not in the top 4. Fix the search side (chunk size, `fetch_k`, hybrid search).
- **ANSWER with the right page retrieved:** read the answer. If it is actually correct, the keywords were too strict. If it is wrong, the model or prompt needs work (try a bigger `MODEL`).
- **Off-topic answered instead of refused:** raise `MIN_SCORE`.
- **LOW GROUNDING:** the model's own answer used words that are not in the PDF text, so the fallback replaced it. The line shows the final answer the user sees. Read it: is the quote still a good answer?

Don't change the test keywords just to make a number go up. Fix a keyword only when the answer is truly correct.

In [ ]:
def show_failures(label=None):
    label = label or list(EVAL_LOG)[-1]      # default: the latest run
    rows = EVAL_LOG[label]["rows"]
    bad = [r for r in rows if not r["ans_ok"] or r["retr_ok"] is False]
    print(f"{len(bad)} problem(s) in [{label}]\n")
    for r in rows:
        if r["grounding"] < GROUND_MIN:
            print(f"⚠️ LOW GROUNDING of the model's own answer ({r['grounding']:.2f}), fallback: {r['fallback']} | {r['q']}")
            print(f"   final answer shown: {r['answer'][:250]}\n")
    for r in bad:
        kind = "RETRIEVAL" if r["retr_ok"] is False else "ANSWER"
        print(f"❌ {kind} | {r['q']}")
        print(f"   expected pages {r['pages_ok'] or 'n/a (off-topic)'} | retrieved pages {r['got_pages']}")
        print(f"   answer: {r['answer'][:250]}\n")

def compare_runs():
    print(f"{'run':<26}{'retrieval':>10}{'answers':>9}{'refusals':>10}{'overall':>9}{'low-ground':>11}{'sec':>6}")
    for label, d in EVAL_LOG.items():
        s = d["summary"]
        print(f"{label:<26}{s['retrieval_hit']:>10.0%}{s['answer_acc']:>9.0%}{s['refusal_rate']:>10.0%}{s['overall']:>9.0%}{s.get('low_grounded', 0):>11}{s.get('seconds', 0):>6}")

show_failures()

1 problem(s) in [llama3.1:8b, temp 0]

❌ ANSWER | What abilities do augmentation modules add to newer RAG implementations?
   expected pages [3] | retrieved pages [2, 2, 1, 3]
   answer: Augmentation modules add the ability to update the external knowledge base with new information without retraining the model.



# Part 8 — The quiz tool

### 8.1 Make quizzes that check themselves
A 3B model often writes questions with a wrong answer key. Every question has to pass these checks before you see it:
1. **Format:** exactly 4 different options and an answer letter A-D.
2. **Evidence:** the model must copy a sentence from the text that proves the answer. We check that this sentence really exists in the text. This catches invented facts. The evidence also becomes the explanation, so the answer and its explanation can no longer disagree.
3. **Answer-key check:** the correct option is the one that shares the most key words with the evidence sentence, and clearly more than any other option. The bot **sets the answer letter from this**, because in your last run the model often wrote the right option but labeled the wrong letter (all 6 "support" rejections were that). If no option clearly wins, the question is dropped.
4. **The question must be about the evidence:** the key words of the question must appear in the evidence sentence. This catches questions like "What is the main goal of the RAG technique?" that were "answered" by an unrelated sentence ("LLMs with RAG are programmed to prioritize new information").
5. **Only one answer is true:** each of the 4 options is checked on its own ("is this a true answer to the question, according to the text?", temperature 0). The check reads the passage **and the other chunks most related to the question**. Exactly one option, the answer key, must be true. If another option is also true, the question is unfair and is dropped. Your screenshot showed this problem: "What can RAG models struggle with?" had the key "Determining accurate sources", but "Combining outdated and updated information" is also true in the PDF, so a player who chose it would be marked wrong. Asking about each option separately is more reliable for a small model than asking it to list all the true ones.
6. **Not too easy:** if two or more options contain "always" or "never" (like "They never struggle with conflicting information"), the question is dropped, and the prompt now asks for plausible wrong options. Your screenshots showed obviously silly wrong answers.
7. **Each fact only once:** two questions built on the same sentence of the PDF (for example, "worst-case outcome" and "what can happen if sources are combined") are counted as one, and the second is dropped.

The explanation shown after you answer is the **whole sentence** from the PDF. The model sometimes copies only half of it, which showed up as a sentence cut off at a comma.

**One question per passage.** In your last run the model returned only about one question per request when asked for several, so most quizzes came back empty. Now the bot asks for **one question from one passage** at a time (up to 3 tries per wanted question) and collects the ones that pass. You may still get fewer questions than requested. If so, the message says how many were rejected and why, and `show_rejects()` shows the actual rejected questions.

`QUIZ_MODEL` is the model used to write and check quizzes. It follows `MODEL` (the 8B model from 1.4). To use a different model just for quizzes, `!ollama pull` it and write its name in the cell below.
For a topic quiz, only chunks that really match the topic are used (the old version used the top 4 even if they were off-topic).

In [ ]:
import random, json, itertools

QUIZ_MODEL = MODEL     # same as MODEL; write another model name here to use it only for quizzes
QUIZ_REJECTS = []      # the questions the last quiz rejected, and why (for debugging)

def quiz_source_texts(topic=None, n=6):
    if topic:
        return [h["text"] for h in relevant_hits(topic, top_k=n)]
    return [c["text"] for c in random.sample(chunks, min(n, len(chunks)))]

def questions_from_passage(passage, avoid=()):
    """Ask for ONE question about ONE passage (small models do this far more reliably than 5 at once)."""
    avoid_text = ("\nDo NOT repeat these questions:\n" + "\n".join(f"- {a}" for a in avoid) + "\n") if avoid else ""
    prompt = f"""Using ONLY the passage below, write ONE multiple-choice question about one of its ideas.{avoid_text}
Return JSON in exactly this format:
{{"question": "...", "options": ["...", "...", "...", "..."], "answer": "A", "evidence": "..."}}
Rules:
- exactly 4 different options
- "answer" is one letter: A, B, C or D
- only one option is correct
- "evidence" is ONE sentence copied word for word from the passage that proves the answer
- each option is a short phrase (under 15 words), and the options do NOT all start with the same words
- the wrong options sound plausible (something a student might believe) but are false according to the passage
- do not use the words always, never, all or none in the options

PASSAGE:
{passage}"""
    reply = ollama.chat(model=QUIZ_MODEL, messages=[{"role": "user", "content": prompt}],
                        format="json", options={"temperature": 0.5, "num_ctx": 4096})
    data = json.loads(reply["message"]["content"])
    return data["questions"] if isinstance(data.get("questions"), list) else [data]

def statement_supported(statement, question, text):
    """Ask ONE yes/no question: is this answer true according to the text? (More reliable than asking for a list.)"""
    prompt = f"""Read the text. Is the ANSWER below a true answer to the QUESTION, according to the text?
Reply with JSON only: {{"supported": true}} or {{"supported": false}}.

TEXT:
{text}

QUESTION: {question}
ANSWER: {statement}"""
    try:
        reply = ollama.chat(model=QUIZ_MODEL, messages=[{"role": "user", "content": prompt}],
                            format="json", options={"temperature": 0, "num_ctx": 4096})
        return json.loads(reply["message"]["content"]).get("supported") is True
    except Exception:
        return False

def true_options(q, text):
    """The set of option letters that are TRUE according to the text (each option is checked on its own)."""
    return {L for L, o in zip("ABCD", q["options"]) if statement_supported(o, q["question"], text)}

QUIZ_KEYS_FIXED = 0
ABSOLUTE = re.compile(r"\b(always|never)\b", re.I)    # silly distractors like "They never struggle"

def fix_answer_key(q):
    """Trust the evidence sentence, not the model's letter. The correct option is the one that matches
    the evidence best. If one option clearly wins, set the key to its letter (the model often mislabels it)."""
    global QUIZ_KEYS_FIXED
    scores = [grounding(o, q["evidence"]) for o in q["options"]]
    best = max(range(4), key=lambda i: scores[i])
    others = max(s for i, s in enumerate(scores) if i != best)
    if scores[best] < 0.5 or scores[best] <= others:
        return False                      # no clear winner: reject the question
    if "ABCD"[best] != q["answer"]:
        QUIZ_KEYS_FIXED += 1
        q["answer"] = "ABCD"[best]
    return True

def full_sentence(evidence, passage):
    """Show the whole sentence the evidence comes from (the model sometimes copies only half of it)."""
    found = [s for s in split_sentences(passage) if evidence_in_context(evidence, s, min_ratio=0.9)]
    return min(found, key=len) if found else evidence.strip()

def check_context(q, passage):
    """The passage plus the best-matching other passages, so an option that is also true elsewhere in the PDF is noticed."""
    extra = [h["text"] for h in relevant_hits(q["question"], top_k=3) if h["text"] not in passage]
    return passage + ("\n\n" + "\n\n".join(extra) if extra else "")

def make_quiz(num_questions=3, topic=None, max_calls=None, progress_cb=None):
    """One question per passage; each question must pass 5 checks. Returns 0 to num_questions questions.
    progress_cb(done, total) is called after each try (the web page uses it for its progress bar)."""
    global QUIZ_REJECTS, QUIZ_KEYS_FIXED
    QUIZ_REJECTS, QUIZ_KEYS_FIXED = [], 0
    max_calls = max_calls or num_questions * 3
    passages = quiz_source_texts(topic, n=max(6, num_questions * 2))
    passages = [p for p in passages if len(p.split()) >= 40] or passages     # very short passages give poor questions
    random.shuffle(passages)

    kept, seen, used_facts = [], set(), set()
    for passage in itertools.islice(itertools.cycle(passages), max_calls):
        try:
            candidates = questions_from_passage(passage, avoid=[k["question"] for k in kept][-10:])
        except (json.JSONDecodeError, KeyError, TypeError):
            QUIZ_REJECTS.append({"reason": "format", "question": "(unreadable JSON)"})
            continue
        for q in candidates:
            if not isinstance(q, dict):
                continue
            q["answer"] = str(q.get("answer", "")).strip().upper()[:1]
            opts, ev = q.get("options"), q.get("evidence", "")
            fmt_ok = (
                isinstance(q.get("question"), str)
                and isinstance(opts, list) and len(opts) == 4
                and all(isinstance(o, str) and 0 < len(o.split()) <= 25 for o in opts)
                and len({o.strip().lower() for o in opts}) == 4
                and q["answer"] in ["A", "B", "C", "D"]
                and isinstance(ev, str)
            )
            true_set = None
            if isinstance(q.get("question"), str) and q["question"].strip().lower() in seen:
                reason = "duplicate"
            elif not fmt_ok:
                reason = "format"
            elif sum(bool(ABSOLUTE.search(o)) for o in opts) >= 2:
                reason = "too-easy"                     # two or more options say "always" / "never"
            elif not evidence_in_context(ev, passage):
                reason = "evidence"
            elif _norm(full_sentence(ev, passage)) in used_facts:
                reason = "same-fact"                    # another question already uses this sentence
            elif grounding(q["question"], ev) < 0.4:
                reason = "off-topic"                    # the question is not about the evidence sentence
            elif not fix_answer_key(q):
                reason = "support"
            else:
                true_set = true_options(q, check_context(q, passage))
                reason = None if true_set == {q["answer"]} else "ambiguous"
            if reason:
                QUIZ_REJECTS.append({"reason": reason, "question": str(q.get("question")), "answer": q["answer"],
                                     "options": q.get("options"), "true": sorted(true_set) if true_set else true_set})
            else:
                q["explanation"] = full_sentence(ev, passage)
                seen.add(q["question"].strip().lower())
                used_facts.add(_norm(q["explanation"]))
                kept.append(q)
        if progress_cb:
            progress_cb(min(len(kept), num_questions), num_questions)
        if len(kept) >= num_questions:
            break
    if len(kept) < num_questions:
        print(f"Quiz: {len(kept)}/{num_questions} questions passed all checks. Rejected:",
              dict(Counter(r["reason"] for r in QUIZ_REJECTS)))
    return kept[:num_questions]

def show_rejects(n=4):
    """See which questions were rejected and why (use when a quiz comes back short)."""
    for r in QUIZ_REJECTS[:n]:
        print(f"[{r['reason']}] {r['question']}")
        if r.get("options"):
            print("   options:", r["options"], "| key:", r.get("answer"), "| model says true:", r.get("true"))

def show_quiz(questions, reveal=True):
    for i, q in enumerate(questions, start=1):
        print(f"Q{i}. {q['question']}")
        for letter, opt in zip("ABCD", q["options"]):
            print(f"   {letter}) {opt}")
        if reveal:
            print(f"   ✅ Answer: {q['answer']} | {q.get('explanation', '')}")
        print()

### 8.2 Test the quiz tool
**Check:** you get 1-3 questions, each with 4 options, an answer letter, and the supporting sentence. Read them yourself: is the marked answer really correct according to the PDF? A tutor will ask. The automatic checks at the bottom should all be `True`.

In [ ]:
quiz = make_quiz(num_questions=3)
print("Random quiz, questions made:", len(quiz))
show_quiz(quiz)
if len(quiz) < 3:
    show_rejects()

topic_quiz = make_quiz(num_questions=2, topic="embeddings")
print("Topic quiz (embeddings), questions made:", len(topic_quiz))
show_quiz(topic_quiz)
if len(topic_quiz) < 2:
    show_rejects()

all_q = quiz + topic_quiz
print("Answer keys corrected from the evidence:", QUIZ_KEYS_FIXED, "(from the last quiz)")
print("All have 4 options:", all(len(q["options"]) == 4 for q in all_q))
print("All answers valid:", all(q["answer"] in "ABCD" for q in all_q))
print("No duplicate questions:", len({q["question"] for q in all_q}) == len(all_q))

Quiz: 1/3 questions passed all checks. Rejected: {'support': 2, 'ambiguous': 3, 'off-topic': 2, 'same-fact': 1}
Random quiz, questions made: 1
Q1. What type of vector is more compact?
   A) Dense vector
   B) Sparse vector
   C) Word vector
   D) Zero vector
   ✅ Answer: A | Dense vectors, which encode meaning, are more compact and contain fewer zeros.

[support] What enhances similarity scoring?
   options: ['Sparse vectors', 'Dense vectors', 'Dot products', 'Approximate nearest neighbor searches'] | key: C | model says true: None
[ambiguous] Which technique helps the model learn retrieval patterns?
   options: ['Inverse Document Task', 'Pre-training with Masked Language Model', 'Supervised Retriever Optimization', 'Reranking Techniques'] | key: A | model says true: set()
[ambiguous] How can LLMs generate misinformation?
   options: ['Even when using correct sources', 'Only when using incorrect sources', 'By pulling information from multiple sources', 'By misinterpreting the context o

# Part 9 — The summary tool

### 9.1 Summaries: whole PDF or one topic
- **Whole PDF = map-reduce.** Summarize each page in 1-2 sentences (map), then summarize those mini-summaries into the final one (reduce). A small model can't read everything at once.
- **One topic:** use search to find the chunks about the topic, then summarize only those.

Corrections:
- The prompt says "start directly with the summary" and "don't expand acronyms". The old topic summary invented "Retrieval-Augmented *Language* Model".
- A small cleaner removes lines like "Here is a summary:" that the model likes to add.
- Topic summaries use only chunks close to the best match. If the PDF only mentions the topic in passing (best rerank score below `WEAK_TOPIC`), the bot quotes the closest text instead of inventing a summary. `WEAK_TOPIC = 2.0` is a starting value: adjust it after you see your scores.

In [ ]:
SUMMARY_CACHE = {}

def tidy_summary(text):
    lines = [l for l in text.strip().split("\n")
             if not (l.strip().lower().startswith("here") and l.strip().endswith(":"))]
    return re.sub(r"\n{3,}", "\n\n", "\n".join(lines)).strip()

def summarize_text(text, style="in 2-3 sentences", focus=None):
    focus_line = f"Only include information about {focus}. Ignore everything else.\n" if focus else ""
    prompt = f"""Summarize the text below {style}.
{focus_line}Use ONLY the information in the text. Do not add anything new.
Do not expand acronyms unless the text does.
Start directly with the summary. Do not write an introduction line like "Here is a summary".

TEXT:
{text}"""
    reply = ollama.chat(model=MODEL, messages=[{"role": "user", "content": prompt}],
                        options={"temperature": 0.2, "num_ctx": 4096})
    return tidy_summary(reply["message"]["content"])

def summarize_pdf(final_style="as one short paragraph, then 4 bullet points of key ideas"):
    page_summaries = []
    for p in pages:                                            # MAP: one mini-summary per page
        s = summarize_text(plain(p["text"]), style="in 1-2 sentences")
        page_summaries.append(f"Page {p['page']}: {s}")
        print(f"  summarized page {p['page']}/{len(pages)}")
    combined = "\n".join(page_summaries)                       # REDUCE: summarize the summaries
    return summarize_text(combined, style=final_style), page_summaries

WEAK_TOPIC = 2.0   # best rerank score below this = the PDF only touches the topic

def summarize_topic(topic, top_k=4):
    hits = search(topic, top_k=top_k)
    if not hits or hits[0]["score"] < MIN_SCORE:
        return "I don't know. This topic isn't covered in the PDF."
    best = hits[0]["rerank"]
    used = [h for h in hits if h["rerank"] >= max(REL_MIN, best - 5)] or hits[:1]   # close to the best match
    text = "\n\n".join(h["text"] for h in used)
    if best < WEAK_TOPIC:                                    # barely mentioned: do not invent a summary
        quote = extract_support(f"What does the text say about {topic}?", text)
        if quote:
            return f"The PDF only mentions this briefly. Closest text: “{quote}”"
        return IDK_SOFT
    return summarize_text(text, style="in 3-4 sentences", focus=topic)

### 9.2 Test the summaries
**Check:**
- The full summary has no "Here is..." line and mentions only things from the PDF.
- The topic summary is about the topic and nothing else.
- The off-topic request says "I don't know".
- All four automatic checks print `True`.
The full summary takes about a minute, and the result is saved so the agent can reuse it instantly.

In [ ]:
full_summary, page_sums = summarize_pdf()
SUMMARY_CACHE["__full__"] = full_summary
print("\n=== FULL SUMMARY ===\n")
print(full_summary)

print("\n=== TOPIC: vector database ===\n")
print(summarize_topic("vector database"))
print("\n=== OFF-TOPIC: football World Cup ===\n")
print(summarize_topic("football World Cup"))

print("\nPage summaries made:", len(page_sums), "(pages:", len(pages), ")")
print("Shorter than the PDF:", len(full_summary.split()) < sum(len(p["text"].split()) for p in pages))
print("No empty page summaries:", all(len(s.split()) > 5 for s in page_sums))
print("No 'Here is' intro line:", not full_summary.lower().startswith("here"))

  summarized page 1/6
  summarized page 2/6
  summarized page 3/6
  summarized page 4/6
  summarized page 5/6
  summarized page 6/6

=== FULL SUMMARY ===

RAG is a technique that enables large language models to retrieve and incorporate new information from external data sources, improving their performance and reducing the risk of providing incorrect information.

• RAG enhances large language models by incorporating an information-retrieval mechanism that allows models to access and utilize additional data beyond their original training set.
• The RAG process involves combining external documents and user input into an LLM prompt to get tailored output, and feeding retrieved information into the LLM to generate output.
• Improvements to the RAG process can be made at different stages, such as optimizing vector similarities and using hybrid vector approaches.
• RAG systems have limitations, including the potential for hallucinations, model retraining needs, and the risk of retrieving 

### 9.3 Check the `WEAK_TOPIC` setting
In your last run, "vector database" got a normal summary (best rerank score 4.4) although the PDF has no section about it. This cell shows the best rerank score for several topics.
**What your run showed:** chunking 8.2, hybrid search 8.4, prompt stuffing 7.7, **vector database 6.2**, embeddings 3.4, reranking 3.1, football -9.8. A one-word search does not separate "has its own section" from "mentioned in passing" (embeddings, which the PDF explains, scores lower than vector database, which it barely mentions). So `WEAK_TOPIC = 2.0` only catches topics that are clearly absent, and the "off-topic" guard already does that. **Leave it at 2.0.** To stop broad summaries instead, the summary prompt now says "only include information about the topic".

In [ ]:
for t in ["chunking", "hybrid search", "embeddings", "reranking", "prompt stuffing", "vector database", "football"]:
    h = search(t)[0]
    print(f"{t:<18} best rerank {h['rerank']:6.2f} | page {h['page']}")
print("\nWEAK_TOPIC is", WEAK_TOPIC, "- topics scoring below it get a quote instead of a summary")

chunking           best rerank   8.21 | page 5
hybrid search      best rerank   8.38 | page 5
embeddings         best rerank   3.38 | page 2
reranking          best rerank   3.06 | page 4
prompt stuffing    best rerank   7.66 | page 2
vector database    best rerank   6.16 | page 5
football           best rerank  -9.76 | page 1

WEAK_TOPIC is 2.0 - topics scoring below it get a quote instead of a summary


# Part 10 — The agent (Agentic AI)

### 10.1 The tools
The agent needs every tool to take the same inputs and return plain text, so each function is wrapped:
- `tool_answer`: the RAG answer with page numbers.
- `tool_quiz`: shows only the **questions** and keeps the quiz in `LAST_QUIZ`.
- `tool_reveal`: shows the answer key of the last quiz (when you type "show answers").
- `tool_summary`: whole-PDF or topic summary, with caching.

**Correction (a real bug):** the router often returned the topic `"PDF"` for "Summarize the PDF". The summary tool then searched for the word "PDF", found nothing, and answered "I don't know". The new `clean_topic()` turns words like *PDF, document, file, it, everything* into "no topic", so the whole document gets summarized.

In [ ]:
LAST_QUIZ = []

GENERIC_TOPIC = re.compile(
    r"(the |this |that |my )?(whole |entire |full )?(pdf|document|doc|file|text|paper|article|content|notes)s?", re.I)
GENERIC_WORDS = {"it", "this", "that", "everything", "all", "summary", "quiz", "questions", "mcq", "mcqs"}
COUNT_TOPIC = re.compile(r"\d+\s*(practice\s+|quiz\s+)?(questions?|mcqs?)", re.I)

MAX_QUIZ = 20
NUM_WORDS = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7, "eight": 8, "nine": 9,
             "ten": 10, "twelve": 12, "fifteen": 15, "twenty": 20}

def parse_count(message, default=3):
    """How many questions did the user ask for? Returns (number, asked_for_it, was_capped)."""
    m = re.search(r"\b(\d{1,3}|" + "|".join(NUM_WORDS) + r")\b\s+(?:\w+\s+){0,2}?(?:questions?|mcqs?|qs)\b", message.lower())
    if not m:
        return default, False, False
    v = m.group(1)
    n = int(v) if v.isdigit() else NUM_WORDS[v]
    return max(1, min(n, MAX_QUIZ)), True, n > MAX_QUIZ

def clean_topic(topic):
    t = (topic or "").strip().strip(".?!\"'").strip()
    if not t or GENERIC_TOPIC.fullmatch(t) or COUNT_TOPIC.fullmatch(t) or t.lower() in GENERIC_WORDS:
        return ""
    return t

def quiz_questions_text(questions):
    lines = []
    for i, q in enumerate(questions, start=1):
        lines.append(f"Q{i}. {q['question']}")
        for letter, opt in zip("ABCD", q["options"]):
            lines.append(f"   {letter}) {opt}")
        lines.append("")
    return "\n".join(lines)

def answers_text(questions):
    return "\n".join(f"Q{i}: ✅ {q['answer']} | {q.get('explanation', '')}"
                     for i, q in enumerate(questions, start=1))

def tool_answer(message, topic=""):
    r = answer(message)
    pages_txt = f"\n(Pages: {r['pages']})" if r["pages"] else ""
    warn = ""
    if r["grounding"] < GROUND_MIN:
        warn = "\n⚠️ Low overlap with the PDF text. Part of this answer may come from the model's general knowledge."
    return r["answer"] + pages_txt + warn

def tool_quiz(message, topic=""):
    global LAST_QUIZ
    if topic:                                    # the topic must exist in the PDF
        hits = search(topic, top_k=1)
        if not hits or hits[0]["score"] < MIN_SCORE:
            LAST_QUIZ = []
            return "I can't quiz you on that. It isn't covered in the PDF."
    n, asked, capped = parse_count(message, default=2 if topic else 3)
    LAST_QUIZ = make_quiz(num_questions=n, topic=topic or None)
    if not LAST_QUIZ:
        return "Sorry, I couldn't make a quiz this time. Try again."
    note = f"(I can make at most {MAX_QUIZ} questions at a time.)\n" if capped else ""
    if len(LAST_QUIZ) < n:
        note += f"(Only {len(LAST_QUIZ)} of {n} questions passed all checks.)\n"
    return (note + quiz_questions_text(LAST_QUIZ)
            + "\nType 'show answers' when you're ready. For a quiz with scoring, use the 📝 Quiz tab.")

def tool_reveal(message, topic=""):
    if not LAST_QUIZ:
        return "There's no quiz yet. Say 'quiz me' first."
    return answers_text(LAST_QUIZ)

def tool_summary(message, topic=""):
    key = topic.lower() if topic else "__full__"      # reuse earlier summaries
    if key not in SUMMARY_CACHE:
        SUMMARY_CACHE[key] = summarize_topic(topic) if topic else summarize_pdf()[0]
    return SUMMARY_CACHE[key]

TOOL_FUNCTIONS = {"answer": tool_answer, "quiz": tool_quiz, "summary": tool_summary, "reveal": tool_reveal}
print("Tools ready:", list(TOOL_FUNCTIONS))
print("clean_topic tests:", [clean_topic(t) for t in ["PDF", "the whole document", "embeddings", "it", "vector databases"]])

Tools ready: ['answer', 'quiz', 'summary', 'reveal']
clean_topic tests: ['', '', 'embeddings', '', 'vector databases']


### 10.2 The router (the model picks the tool)
**Correction:** in your last run, "show answers" typed when no quiz existed was sent to the summary tool and printed a whole summary. Short commands like "show answers" now always go to `reveal`, which replies "There's no quiz yet".

- The prompt describes the tools and gives examples. Small models follow examples much better than rules.
- `temperature: 0` makes the choice consistent, and `format="json"` forces valid JSON.
- Two shortcuts run first or last: if a quiz exists and you ask for answers, it goes straight to `reveal`. If the model's output is broken or names a tool that doesn't exist, simple keyword rules take over (`rule_route`). An agent should not crash because the model made a mistake.
- The agent makes **exactly one** tool call per message, so it can never get stuck in a loop.

**Cleanup:** the old notebook defined `route` twice. There is only one now.

In [ ]:
ROUTER_PROMPT = """You are a router for a study assistant. Pick ONE tool for the user's message.

Tools:
- answer: the user asks a question about the document's content
- quiz: the user wants to be tested or quizzed
- summary: the user wants a summary or overview

Return JSON only: {"tool": "answer", "topic": ""}
"topic" is a specific subject the user mentioned (like "embeddings"), or "" if they mean the whole document.

Examples:
Message: What is chunking? -> {"tool": "answer", "topic": ""}
Message: Test me on vector databases -> {"tool": "quiz", "topic": "vector databases"}
Message: Give me 3 MCQs -> {"tool": "quiz", "topic": ""}
Message: Quiz me with 10 questions -> {"tool": "quiz", "topic": ""}
Message: Summarize the whole PDF -> {"tool": "summary", "topic": ""}
Message: Summarize the PDF for me -> {"tool": "summary", "topic": ""}
Message: Give me an overview of embeddings -> {"tool": "summary", "topic": "embeddings"}

Message: """

def rule_route(message):
    m = message.lower()
    if any(k in m for k in ["quiz", "test me", "test my", "mcq", "practice question"]):
        return {"tool": "quiz", "topic": ""}
    if any(k in m for k in ["summar", "overview", "tl;dr", "tldr"]):
        return {"tool": "summary", "topic": ""}
    return {"tool": "answer", "topic": ""}

SHOW_ANSWERS = re.compile(r"^\W*(please )?(show|reveal|see|give)( me)?( the)?( quiz)?( answers| answer key)\W*$|answer key|check my answers")
SHOW_ANSWERS_LOOSE = re.compile(r"(show|reveal|see|give).{0,15}answers\b")

def route(message):
    m = message.lower()
    # short commands like "show answers" always go to reveal (it says "no quiz yet" if there is none)
    if SHOW_ANSWERS.search(m) or (LAST_QUIZ and SHOW_ANSWERS_LOOSE.search(m)):
        return {"tool": "reveal", "topic": ""}
    try:
        reply = ollama.chat(
            model=MODEL,
            messages=[{"role": "user", "content": ROUTER_PROMPT + message}],
            format="json",
            options={"temperature": 0},
        )
        data = json.loads(reply["message"]["content"])
        tool, topic = data.get("tool"), data.get("topic", "")
        if tool in ("answer", "quiz", "summary") and isinstance(topic, str):
            return {"tool": tool, "topic": clean_topic(topic)}
    except Exception:
        pass
    return rule_route(message)

### 10.3 Chat memory, and the agent function
**Chat memory:** "How is it used in RAG?" means nothing to a search engine. `make_standalone` looks at the last few messages and rewrites it as "How is a vector database used in RAG?" before searching. If the rewrite fails or looks wrong, the original message is used. It only helps the **answer** tool.
**The agent:** route → (rewrite if it is a follow-up question) → run the chosen tool → return the result.

In [ ]:
def recent_history_text(history, max_items=6):
    lines = []
    for item in (history or [])[-max_items:]:
        if isinstance(item, dict):                                   # newer Gradio format
            content = item.get("content", "")
            if isinstance(content, list):
                content = " ".join(str(c.get("text", "")) if isinstance(c, dict) else str(c) for c in content)
            lines.append(f"{item.get('role', 'user')}: {str(content)[:300]}")
        elif isinstance(item, (list, tuple)) and len(item) == 2:     # older format
            lines.append(f"user: {str(item[0])[:300]}")
            lines.append(f"assistant: {str(item[1])[:300]}")
    return "\n".join(lines)

def make_standalone(message, history):
    context = recent_history_text(history)
    if not context:
        return message
    prompt = f"""Rewrite the user's NEW MESSAGE as a complete, standalone question.
Replace words like it, this, that, they, these, those with what they refer to in the chat.
If the message already makes sense alone, return it unchanged. Return only the question.

Example:
CHAT:
user: What is chunking?
assistant: Chunking breaks the data into smaller pieces.
NEW MESSAGE: Why is it needed?
STANDALONE QUESTION: Why is chunking needed?

Now your turn.
CHAT:
{context}

NEW MESSAGE: {message}

STANDALONE QUESTION:"""
    try:
        reply = ollama.chat(model=MODEL, messages=[{"role": "user", "content": prompt}],
                            options={"temperature": 0, "num_ctx": 4096})
        rewritten = reply["message"]["content"].strip().strip('"')
        if 3 <= len(rewritten) <= 300:
            return rewritten
    except Exception:
        pass
    return message

def agent(message, history=None, verbose=True):
    decision = route(message)
    tool, topic = decision["tool"], decision["topic"]
    query = message
    if tool == "answer" and history:
        query = make_standalone(message, history)
        if verbose and query != message:
            print(f"🔁 Rewritten: {query}")
    if verbose:
        print(f"🧭 Agent chose: {tool}" + (f" (topic: {topic})" if topic else ""))
    output = TOOL_FUNCTIONS[tool](query, topic)
    return {"tool": tool, "topic": topic, "output": output}

### 10.4 Test the routing alone (fast)
Only the router's choice is tested here, not the tool outputs.
**Check:** 7/8 or better. The off-topic football question should go to **answer**, which then says "I don't know".

In [ ]:
routing_tests = [
    ("What is retrieval-augmented generation?", "answer"),
    ("How does RAG reduce hallucinations?", "answer"),
    ("Quiz me on vector databases", "quiz"),
    ("Can you test my knowledge?", "quiz"),
    ("Summarize the PDF for me", "summary"),
    ("Summarize the PDF", "summary"),
    ("Give me a quick overview of embeddings", "summary"),
    ("Who won the football World Cup in 2018?", "answer"),
    ("Make me 3 practice questions", "quiz"),
]

correct = 0
for msg, expected in routing_tests:
    d = route(msg)
    ok = d["tool"] == expected
    correct += ok
    print(f"{'✅' if ok else '❌'} {msg!r} → {d['tool']}" + (f" (topic: {d['topic']})" if d["topic"] else ""))
print(f"\nRouting accuracy: {correct}/{len(routing_tests)}")
print("Whole-document summary requests keep no topic:",
      route("Summarize the PDF for me")["topic"] == "" and route("Summarize the PDF")["topic"] == "")

✅ 'What is retrieval-augmented generation?' → answer
✅ 'How does RAG reduce hallucinations?' → answer (topic: RAG)
✅ 'Quiz me on vector databases' → quiz (topic: vector databases)
✅ 'Can you test my knowledge?' → quiz
✅ 'Summarize the PDF for me' → summary
✅ 'Summarize the PDF' → summary
✅ 'Give me a quick overview of embeddings' → summary (topic: embeddings)
✅ 'Who won the football World Cup in 2018?' → answer
✅ 'Make me 3 practice questions' → quiz

Routing accuracy: 9/9
Whole-document summary requests keep no topic: True


### 10.5 Test the full agent end to end
**Check:**
1. A short answer with page numbers.
2. A quiz **without** answers, then `show answers` reveals them.
3. A real summary (not "I don't know"), and a second request that returns instantly from the cache.
4. A topic summary about chunking only.
5. The follow-up question is rewritten to mention "vector database".
6. A quiz about football is refused.

In [ ]:
for msg in ["What is a vector database?", "Quiz me on embeddings", "show answers",
            "Summarize the PDF for me", "Summarize chunking"]:
    print("=" * 60)
    print("You:", msg)
    print(agent(msg)["output"])
    print()

history = [{"role": "user", "content": "What is a vector database?"},
           {"role": "assistant", "content": "A vector database stores embeddings so similar text can be found quickly."}]
print("Follow-up rewrite:", make_standalone("How is it used in RAG?", history))
print("Quiz guard:", agent("Quiz me on football", verbose=False)["output"])

t = time.time(); agent("Summarize the PDF", verbose=False); print("Summary again (cached):", round(time.time() - t, 1), "s")

You: What is a vector database?
🧭 Agent chose: answer
I don't know.

You: Quiz me on embeddings
🧭 Agent chose: quiz (topic: embeddings)
Quiz: 0/2 questions passed all checks. Rejected: {'support': 2, 'off-topic': 4}
Sorry, I couldn't make a quiz this time. Try again.

You: show answers
🧭 Agent chose: reveal
There's no quiz yet. Say 'quiz me' first.

You: Summarize the PDF for me
🧭 Agent chose: summary
RAG is a technique that enables large language models to retrieve and incorporate new information from external data sources, improving their performance and reducing the risk of providing incorrect information.

• RAG enhances large language models by incorporating an information-retrieval mechanism that allows models to access and utilize additional data beyond their original training set.
• The RAG process involves combining external documents and user input into an LLM prompt to get tailored output, and feeding retrieved information into the LLM to generate output.
• Improvements to t

# Part 11 — The web page (Gradio)

### 11.1 Install Gradio
**Check:** a version number is printed. If Colab asks you to restart the runtime, do it, then run the notebook again from the top.

In [ ]:
!pip install -q --upgrade gradio
import gradio as gr
print("Gradio version:", gr.__version__)

Gradio version: 6.30.0


### 11.2 The function the page calls
Gradio passes the user's message and the chat history. **Correction:** the old version never passed `history` to the agent, so chat memory did nothing on the web page. Now it does. A `try/except` shows a friendly message instead of crashing the page.

In [ ]:
def chat_fn(message, history):
    if not message or not message.strip():
        return "Please type a question, or ask me to quiz or summarize."
    try:
        result = agent(message, history=history, verbose=False)
        label = {"answer": "💬 Answer", "quiz": "📝 Quiz", "summary": "📄 Summary", "reveal": "✅ Answers"}[result["tool"]]
        return f"**{label}**\n\n{result['output']}"
    except Exception as e:
        return f"Sorry, something went wrong: {e}"

print(chat_fn("What is a vector database?", []))
print("-" * 50)
print(chat_fn("", []))

**💬 Answer**

I don't know.
--------------------------------------------------
Please type a question, or ask me to quiz or summarize.


### 11.3 The quiz page logic (works without Gradio, so it can be tested)
This is the "brain" of the interactive quiz, like the quiz screen in Gemini: one question at a time, tap an answer, see right or wrong at once with the reason, a ❌/✅ counter, Back and Next, and a score at the end.
- `state` is a small dictionary: the list of questions, which question you are on (`i`), and which answers you picked.
- `quiz_view(state)` turns the state into what the page shows: the status line, the question, the choices, and the feedback.
- `quiz_pick` records your answer (you can't change it afterwards). `quiz_move` goes back or forward. Going forward from the last question shows the score and the questions you missed.
- The reason shown after you answer is the **sentence from the PDF** that proves the answer, so the feedback is based on your document.

**Check:** the test lines at the bottom print "True" for every check.

In [ ]:
def quiz_view(st):
    """Everything the quiz page shows for the current state."""
    quiz, i, picked = st["quiz"], st["i"], st["picked"]
    right = sum(1 for k, v in picked.items() if v == quiz[k]["answer"])
    wrong = len(picked) - right
    view = {"status": "", "question": "", "choices": [], "picked": None, "locked": False, "feedback": ""}
    if not quiz:
        view["question"] = "Choose how many questions you want, then press **Generate quiz**."
        return view
    if i >= len(quiz):                                              # results page
        pct = round(100 * right / len(quiz))
        missed = [f"- Q{k + 1}. {q['question']} → **{q['answer']}. {q['options']['ABCD'.index(q['answer'])]}**"
                  for k, q in enumerate(quiz) if picked.get(k) != q["answer"]]
        view["status"] = f"**Finished**   ❌ {wrong}   ✅ {right}"
        view["question"] = (f"## Score: {right} / {len(quiz)} ({pct}%)\n"
                            + ("\n**Review these:**\n" + "\n".join(missed) if missed else "\n🎉 Perfect score!"))
        return view
    q = quiz[i]
    view["status"] = f"**Question {i + 1} / {len(quiz)}**   ❌ {wrong}   ✅ {right}"
    view["question"] = f"### {q['question']}"
    view["choices"] = [(f"{L}. {opt}", L) for L, opt in zip("ABCD", q["options"])]
    letter = picked.get(i)
    if letter:
        view["picked"], view["locked"] = letter, True
        why = f"\n\n**Why (from the PDF):** {q.get('explanation', '')}"
        if letter == q["answer"]:
            view["feedback"] = "✅ **Correct!**" + why
        else:
            view["feedback"] = (f"❌ **Not quite.** The correct answer is **{q['answer']}. "
                                f"{q['options']['ABCD'.index(q['answer'])]}**" + why)
    return view

def quiz_pick(st, letter):
    i = st["i"]
    if st["quiz"] and i < len(st["quiz"]) and letter in list("ABCD") and i not in st["picked"]:
        st["picked"][i] = letter
    return st

def quiz_move(st, delta):
    st["i"] = max(0, min(st["i"] + delta, len(st["quiz"])))
    return st

# --- tests (no web page needed) ---
demo_quiz = make_quiz(num_questions=3)
st = {"quiz": demo_quiz, "i": 0, "picked": {}}
v = quiz_view(st)
print("Quiz made:", len(demo_quiz), "questions | first view:", v["status"])
if demo_quiz:
    wrong_letter = next(L for L in "ABCD" if L != demo_quiz[0]["answer"])
    quiz_pick(st, wrong_letter)
    print("Wrong pick shows feedback:", quiz_view(st)["feedback"].startswith("❌"), "| locked:", quiz_view(st)["locked"])
    quiz_pick(st, demo_quiz[0]["answer"])                                  # second pick is ignored
    print("Answer can't be changed:", st["picked"][0] == wrong_letter)
    for _ in range(len(demo_quiz)):
        quiz_move(st, +1)
    print("Results page:", quiz_view(st)["question"].split("\n")[0])
print("parse_count tests:", [parse_count(t)[:2] for t in
      ["quiz me with 10 questions", "give me 5 MCQs", "make me ten practice questions", "quiz me on chunking", "50 questions"]])

Quiz made: 3 questions | first view: **Question 1 / 3**   ❌ 0   ✅ 0
Wrong pick shows feedback: True | locked: True
Answer can't be changed: True
Results page: ## Score: 0 / 3 (0%)
parse_count tests: [(10, True), (5, True), (10, True), (3, False), (20, True)]


### 11.4 Launch the page (two tabs: Chat and Quiz)
- **Chat tab:** the same agent as before (questions, summaries, text quizzes).
- **Quiz tab:** pick **how many questions** (1 to 20) with the slider, optionally a topic, and press Generate. A progress bar shows while the questions are made (about 3-5 seconds per question on the T4 GPU, so 20 questions take 1-2 minutes). Then answer one by one like in Gemini.

`share=True` creates a public `.gradio.live` link that works **only while this notebook is running** and expires after about a week. Anyone with the link can use your Colab, so share it only with your tutor. Run the last cell to stop it.

**Known limits:** the text-quiz answers ("show answers") and the summary cache are shared by everyone using the link. The Quiz tab keeps each visitor's quiz separate. This page code is the one part I could not run here, so if it errors, send me the error message.

In [ ]:
def quiz_outputs(st, note=""):
    v = quiz_view(st)
    feedback = (note + "\n\n" + v["feedback"]).strip() if note else v["feedback"]
    radio = gr.Radio(choices=v["choices"], value=v["picked"], interactive=not v["locked"],
                     label="Choose an answer", visible=bool(v["choices"]))
    return st, v["status"], v["question"], radio, feedback

def on_generate(n, topic, progress=gr.Progress()):
    n, topic = int(n), clean_topic(topic or "")
    empty = {"quiz": [], "i": 0, "picked": {}}
    if topic:
        hits = search(topic, top_k=1)
        if not hits or hits[0]["score"] < MIN_SCORE:
            return quiz_outputs(empty, "I can't quiz you on that. It isn't covered in the PDF.")
    qs = make_quiz(num_questions=n, topic=topic or None,
                   progress_cb=lambda done, total: progress(done / total, desc=f"{done}/{total} questions ready"))
    if not qs:
        return quiz_outputs(empty, "I couldn't make a quiz this time. Try again.")
    note = f"Only {len(qs)} of {n} questions passed all checks." if len(qs) < n else ""
    return quiz_outputs({"quiz": qs, "i": 0, "picked": {}}, note)

def on_pick(letter, st):
    return quiz_outputs(quiz_pick(st, letter))

def on_next(st):
    return quiz_outputs(quiz_move(st, +1))

def on_back(st):
    return quiz_outputs(quiz_move(st, -1))

with gr.Blocks(title="PDF Study Buddy") as demo:
    gr.Markdown("# 📚 PDF Study Buddy")
    with gr.Tab("💬 Chat"):
        gr.ChatInterface(
            fn=chat_fn,
            description="Ask questions about the PDF, or say 'summarize'. The agent picks the right tool automatically.",
            examples=["What is retrieval-augmented generation?", "Quiz me with 5 questions",
                      "Summarize the PDF for me", "Who won the football World Cup in 2018?"],
        )
    with gr.Tab("📝 Quiz"):
        with gr.Row():
            n_slider = gr.Slider(1, MAX_QUIZ, value=5, step=1, label="How many questions?")
            topic_box = gr.Textbox(label="Topic (optional)", placeholder="e.g. chunking. Leave empty for the whole PDF.")
        start_btn = gr.Button("Generate quiz", variant="primary")
        status_md = gr.Markdown()
        question_md = gr.Markdown("Choose how many questions you want, then press **Generate quiz**.")
        radio = gr.Radio(choices=[], label="Choose an answer", visible=False)
        feedback_md = gr.Markdown()
        with gr.Row():
            back_btn = gr.Button("◀ Back")
            next_btn = gr.Button("Next ▶", variant="primary")
        quiz_state = gr.State({"quiz": [], "i": 0, "picked": {}})

        outs = [quiz_state, status_md, question_md, radio, feedback_md]
        start_btn.click(on_generate, [n_slider, topic_box], outs)
        radio.input(on_pick, [radio, quiz_state], outs)
        next_btn.click(on_next, [quiz_state], outs)
        back_btn.click(on_back, [quiz_state], outs)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ef6004d43bbe14e801.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### 11.5 Stop the page (when you are done)

In [ ]:
demo.close()

Closing server running on port: 7860


# What to say when you present this

**In one sentence:** "A study assistant that reads a PDF and can answer questions, quiz you and summarize, using a local model; an agent picks the right tool."

**The three parts:**
- **RAG:** chunks and embeddings in Pinecone, a reranker, and an answer that uses only retrieved text.
- **Gen AI:** a local Ollama model writes answers, quizzes and summaries.
- **Agentic AI:** a router decides which tool to use, with a keyword fallback and a one-call limit.

**Accuracy:** you can show real numbers from Part 7 (retrieval hit rate, answer accuracy, off-topic refusal), and say how you improved them.

**Honest limits:**
- The grounding fallback quotes the PDF, but the quote may only be loosely related to a question the PDF barely covers. That is better than an invented answer, and it is shown as "Closest text in the PDF".
- The thresholds (`MIN_SCORE`, `REL_MIN`, `GROUND_MIN`, `WEAK_TOPIC`) are set from this one PDF and a few dozen questions. Re-check them for a new PDF.
- A quiz may contain fewer questions than requested, because questions that fail the checks are dropped. Even with the checks, a small model can still write a weak question, so read quizzes before using them.
- The evaluation is keyword-based, so failures are read by hand, and the question set is written for this one PDF.
- Heading detection is a simple rule (short lines with letters only). It fits Wikipedia-style PDFs and may need `DETECT_HEADINGS = False` for others.
- Text inside a PDF is trusted as context. A malicious PDF could contain instructions aimed at the model.
- Quiz and summary state is shared by everyone using the public link.